[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_3_Describing_a_Column.ipynb)

# 06.3: Describing a Column

Notebook 06.2 picked out particular majors. Often you want a sense of the whole column first: what is a typical value, and how much do the values vary? A list of 173 numbers is too much to take in by reading it, so we use **descriptive statistics**, which are numbers that summarize a column. pandas calculates them for you, so this notebook is about what each one tells you and when it can mislead you. We will not work through the formulas.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

This uses the majors data again, with `salaries`, `share_women` and a new series, `category`, which gives the broad field that each major belongs to. Keep in mind what one value of `salaries` is: the median salary for one major. Every major counts once, whether it has hundreds of graduates or hundreds of thousands, so the statistics below describe a typical major and not a typical graduate.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
category = pd.Series(majors["Major_category"].to_list(), index=major_list)

## What is a typical value? Mean, median and mode

There are three common answers to this question. The **mean** is the average: add up the values and divide by how many there are. The **median** is the middle value when the values are in order, so half are below it and half are above it. The **mode** is the value that appears most often.

In [ ]:
print(salaries.mean())
print(salaries.median())
print(salaries.mode())

For a typical major, the mean of these medians is about 40151 and the median of them is 36000. They differ because some salaries are far above the rest, such as PETROLEUM ENGINEERING at 110000. Every value counts toward the mean, so a very large value pulls the mean up. The median only depends on which value is in the middle, so a very large value barely moves it. The mode is 35000, which 20 majors share. `.mode()` returns a series and not a single number, because several values can tie for most common. A small example makes the difference between the mean and the median easier to see.

In [ ]:
small = pd.Series([30, 30, 40, 50, 1000])
print(small.mean())
print(small.median())

The mean of 230 is higher than four of the five values, and it does not describe any of them well. The median of 40 is much closer to a typical value. When a column has a long tail of very large or very small values, as these salaries do on the high side, the data is called **skewed**, and the median usually gives a better picture of a typical value. When the mean and the median are close together, either one is a reasonable summary.

## Smallest, largest, how many and the total

`.min()`, `.max()`, `.count()` and `.sum()` do what their names say. `.count()` counts values that are not missing, and we will come back to why that matters. Subtracting the minimum from the maximum gives the **range**, which is the simplest measure of how spread out the values are.

In [ ]:
print(salaries.min())
print(salaries.max())
print(salaries.count())
print(salaries.max() - salaries.min())

Median salaries run from 22000 to 110000 across 173 majors, which is a range of 88000. The range depends entirely on the two most extreme values, so one unusual major can make it very large.

## How spread out are the values? Standard deviation and percentiles

Two columns can have the same mean and look very different, so we also want a better measure of spread. The **standard deviation** describes roughly how far values tend to be from the mean. A larger number means the values are more spread out, and it is in the same units as the data.

In [ ]:
salaries.std()

The standard deviation is about 11470, in dollars, against a mean of about 40151. About 80 percent of these majors are within one standard deviation of the mean. Because it is based on the mean, the standard deviation is also pulled up by very large values.

A **percentile** gives the value that a certain percentage of the data falls below. The 25th percentile is the value with 25 percent of the data below it, and the 50th percentile is the same thing as the median. `.quantile()` takes the percentile as a fraction.

In [ ]:
print(salaries.quantile(0.25))
print(salaries.quantile(0.75))

About a quarter of the majors have a median salary below 33000, and about a quarter are above 45000, so roughly the middle half of majors falls between 33000 and 45000. It is not exactly a quarter because many majors share the same value. That range ignores the extreme values at either end, so unlike the standard deviation it does not change if you remove PETROLEUM ENGINEERING.

## All of them at once: `.describe()`

`.describe()` produces most of these numbers in one call, and it is usually the first thing to run on a numeric column.

In [ ]:
salaries.describe()

Read it from the top. `count` is 173 values, `mean` is about 40151 and `std` is about 11470. `min` and `max` are 22000 and 110000. The rows labeled 25%, 50% and 75% are the percentiles from above, so the median appears here as the 50% row. You can check the shape from these numbers: the max of 110000 is much farther from the median of 36000 than the min of 22000 is, which points to a long tail of high salaries.

## Text columns: counting categories

A mean makes no sense for a column like `category`. For a column of categories, the useful summary is how many times each value appears, which is what `.value_counts()` gives you. It returns a series where each distinct value is a label and its count is the value, sorted from most to least common.

In [ ]:
category.value_counts().head()

Engineering is the most common category with 29 majors, followed by Education with 16. The last line of the output says `Name: count`, which is just pandas labeling what the numbers are. Passing `normalize=True` gives fractions instead of counts, so 29 out of 173 becomes about 0.168, or about 17 percent of the majors.

In [ ]:
category.value_counts(normalize=True).head(2)

`.unique()` lists each distinct value once, and `.nunique()` counts them.

In [ ]:
print(category.unique()[:4])
print(category.nunique())

The first four distinct categories, in the order they first appear, are Engineering, Business, Physical Sciences and Law & Public Policy, and there are 16 categories in all. Here is a related use of what you know about masks. Since `True` counts as 1 and `False` as 0, the mean of a boolean series is the fraction of values that are `True`.

In [ ]:
(salaries > 50000).mean()

About 0.145, or 14.5 percent of the majors, have a median above $50,000. That is the 25 majors you saw in 06.2, divided by 173.

## Missing values

Real data often has gaps. pandas marks a missing value as `NaN`, short for "not a number". Compare the length of `share_women` with its count.

In [ ]:
print(len(share_women))
print(share_women.count())

The length is 173 but the count is 172, so one value is missing. `.isna()` makes a mask that is `True` where a value is missing, so you can both count the missing values and find them.

In [ ]:
print(share_women.isna().sum())
share_women[share_women.isna()]

FOOD SCIENCE is the one missing share. The statistics in this notebook all skip missing values automatically, so `share_women.mean()` averages the 172 values that exist. That is usually what you want, but it means a summary can silently leave out rows, so it is worth comparing `count` with the length of a series. Notebook 07.3 looks at what to do about missing values.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
category = pd.Series(majors["Major_category"].to_list(), index=major_list)
unemployment = pd.Series(majors["Unemployment_rate"].to_list(), index=major_list)

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
`unemployment` holds the unemployment rate for each major, as a fraction. Store its mean in a variable named `mean_rate` and its median in a variable named `median_rate`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "mean_rate", unemployment.mean(), "Use .mean() on unemployment.")
check(1, "median_rate", unemployment.median(), "Use .median() on unemployment.")

### Problem 2
Store the standard deviation of `unemployment` in a variable named `spread`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "spread", unemployment.std(), "Standard deviation has its own method.")

### Problem 3
Store the 90th percentile of `salaries` in a variable named `pay_90`.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "pay_90", salaries.quantile(0.9), ".quantile() takes the percentile as a fraction.")

### Problem 4
Create a series named `category_shares` that gives the fraction of majors in each category.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "category_shares", category.value_counts(normalize=True), "Use .value_counts() with normalize=True.")

### Problem 5
Store the number of different categories in a variable named `category_total`.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "category_total", category.nunique(), ".unique() lists the values, and another method counts them.")

### Problem 6
Store the fraction of majors with a median salary below 30000 in a variable named `low_share`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "low_share", (salaries < 30000).mean(), "Make a mask and then take its mean, since True counts as 1.")

### Problem 7
Store the number of missing values in `share_women` in a variable named `missing_count`.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "missing_count", share_women.isna().sum(), ".isna() makes a mask that you can add up.")

## What's next

You can now summarize a numeric column and a column of categories. All of this treats text as whole values to count, though. Often the information you want is inside the text, such as whether a major's name includes the word ENGINEERING. Notebook 06.4 shows how to work with the text in a series.